In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

from agx_core.models.ra.encoder import Encoder
from agx_core.models.ra.decoder import Decoder
from agx_tf.models.ra.model import ReversedAutoencoder

enc = Encoder()

img_size = 192
res = img_size // 2 ** len(enc.filters)

img_shape = (img_size, img_size, 1)
cond_shape = (res, res, 1)

dec = Decoder(target_shape=img_shape)
ra = ReversedAutoencoder(enc, dec, scale=0.05)

In [ ]:
import numpy as np
import tensorflow as tf
import albumentations as A


from pathlib import Path
from PIL import Image


class UnlabeledImageDataset(tf.data.Dataset):
    def __init__(self, root_dir: Path, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        # Get list of all image file names in the folder
        self.image_files = list(root_dir.glob("*.jpg"))

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        image = Image.open(img_name).convert("L")

        if self.transform:
            image = self.transform(image=np.array(image))
            image = image["image"][..., np.newaxis]

        condition = tf.ones((res, res, 1), dtype=tf.int64)
        return (image, condition), image.copy()


def create_generator(path: Path):
    image_files = list(path.glob("*.jpg"))

    def generator():
        for fpath in image_files:
            image = Image.open(fpath).convert("L")
            yield np.array(image)[..., np.newaxis]

    return generator


train_path = Path("../data/products/TunaZenithal/train/")
valid_path = Path("../data/products/TunaZenithal/val/")

tx_train = A.Compose(
    [
        A.InvertImg(1),
        A.Resize(img_size, img_size),
        A.HorizontalFlip(0.5),
        # A.Affine(scale=(0.9, 0.95), rotate=(15, 90), shear=(-2, 2), p=0.5),
        A.RandomRotate90(0.5),
        # A.RandomBrightnessContrast(
        #     brightness_limit=0.2, contrast_limit=(-0.5, 0.5), p=0.5
        # ),
        A.GaussianBlur(blur_limit=(1, 3), p=0.3),
        # A.Normalize(mean=[0.5], std=[0.5]),
        A.ToFloat(255),
    ]
)

tx_valid = A.Compose(
    [
        A.InvertImg(1),
        A.Resize(img_size, img_size),
        A.ToFloat(255),
    ]
)


def make_augment_fn(transform, img_size, channels=1):
    """Wraps an Albumentations Compose into a tf.data-compatible map function."""

    def augment(image):
        def apply_transform(img):
            # img is a numpy array here (via .numpy())
            aug = transform(image=img.numpy())
            return aug["image"].astype(np.float32)

        # tf.py_function bridges the TF graph and Python/NumPy world
        aug_image = tf.py_function(func=apply_transform, inp=[image], Tout=tf.float32)

        # py_function loses static shape info — restore it explicitly
        aug_image.set_shape([img_size, img_size, channels])
        return aug_image

    return augment


def make_format_fn(resolution, channels=1):
    """Formats augmented image into ((image, cond), target) for a custom training step."""

    def format_sample(image):
        cond = tf.ones((resolution, resolution, channels), dtype=tf.int64)
        return (image, cond), image  # target is a dummy copy

    return format_sample


ds_train = tf.data.Dataset.from_generator(
    create_generator(train_path),
    output_signature=tf.TensorSpec(shape=(None, None, 1), dtype=tf.uint8),
)

ds_valid = tf.data.Dataset.from_generator(
    create_generator(valid_path),
    output_signature=tf.TensorSpec(shape=(None, None, 1), dtype=tf.uint8),
)

ds_train = ds_train.map(
    make_augment_fn(tx_train, img_size), num_parallel_calls=tf.data.AUTOTUNE
).map(make_format_fn(res), num_parallel_calls=tf.data.AUTOTUNE)
ds_valid = ds_valid.map(
    make_augment_fn(tx_valid, img_size), num_parallel_calls=tf.data.AUTOTUNE
).map(make_format_fn(res), num_parallel_calls=tf.data.AUTOTUNE)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
it = iter(ds_train)
for idx, ax in enumerate(axes.flat):
    (X, y), _ = next(it)
    ax.imshow(X, cmap="gray")
    ax.set_title(f"Image {idx + 1}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from agx_core.models.ra.optimizer import RAOptimizer

from keras.optimizers import Adam
from keras.losses import  MeanSquaredError

optimizer = RAOptimizer(Adam(learning_rate=1e-9), Adam(learning_rate=6e-6))
loss = MeanSquaredError(reduction=None)

ra.build([img_shape, cond_shape])
ra.compile(optimizer, loss)

In [ ]:
ds_train = ds_train.batch(8).prefetch(tf.data.AUTOTUNE)
ds_valid = ds_valid.batch(8).prefetch(tf.data.AUTOTUNE)

history = ra.fit(ds_train, validation_data=ds_valid, epochs=20)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 12))

for idx, ax in enumerate(axes.flat):
    (I, C), y = ds_valid[idx]
    rec = ra([I[np.newaxis, ...], C[np.newaxis, ...]])
    ax.imshow(rec.cpu()[0], cmap="gray")
    ax.set_title(f"Image {idx + 1}")
    ax.axis("off")

plt.tight_layout()
plt.show()